# 检验的功效函数

> 对应代码：`MathStatsCode/code_in_notes/Chap.12/test_power_function.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对样本量5/30/100分别模拟 t 检验的拒绝概率随真实均值变化的曲线（功效函数）。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.12`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.12")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页（后续用图形与模拟输出）。


In [ ]:
%%stata
clear
set more off


**test under H0: mu=mu0 at 5% significant level**

定义检验程序 `test_normal_mean`（与 test_under_null.do 完全相同）：生成正态样本并计算 t 统计量，返回是否拒绝 H0: μ=μ0 的指示变量。


In [ ]:
%%stata
cap program drop test_normal_mean
program define test_normal_mean, rclass
	version 12
	syntax [,obs(integer 10) mu(real 0) mu0(real 0) sigma(real 1)]
	drop _all
	set obs `obs'
	tempvar x
	gen `x'=`sigma'*rnormal()+`mu'
	quietly: su `x'
	tempname std_mean rejected
	scalar `std_mean'=sqrt(`obs')*(r(mean)-`mu0')/r(sd)
	di `std_mean'
	if abs(`std_mean')>invt(`obs'-1,1-0.05/2){
		scalar `rejected'=1
	}
	else{
		scalar `rejected'=0
	}
	return scalar rejected=`rejected'
end


**replicate test for 1000 times by default**

定义功效计算程序 `compute_power`：`syntax ... reps(1000)` 接受样本量、真实均值、原假设均值与模拟次数；内部先在 `preserve`（保存当前内存数据）之后 `drop _all`、用 `simulate` 重复检验 `reps` 次，求拒绝频率 `r(mean)` 作为**功效**（power）返回；`restore` 再把内存恢复到调用前的状态——`preserve/restore` 是编写“不污染调用者数据”的程序的常用技巧。


In [ ]:
%%stata
cap program drop compute_power
program define compute_power, rclass
	version 12
	syntax [,obs(integer 10) mu(real 0) mu0(real 0)/*
		*/ sigma(real 1) reps(integer 1000)]
	quietly{
	// 缓存内存中现有的变量
	preserve
	drop _all
	simulate rejected=r(rejected),reps(`reps'):/*
		*/ test_normal_mean, obs(`obs') mu(`mu')/*
		*/ mu0(`mu0') sigma(`sigma')
	quietly: su rejected
	return scalar power=r(mean)
	// 恢复preserve之前内存变量
	restore
	}
end


**draw power function for sample size=[5 30 100]**

主程序计算并绘制**功效函数**：`set obs 100` 建立一个100行的空数据集，`gen mu=-3+(_n-1)*6/99` 生成从 -3 到 3 的等距网格（作为真实均值的横坐标）；`tempname mu` 生成一个临时标量名（避免与变量 mu 混淆），预置三列 `power5/power30/power100` 存放三个样本量下的功效。外层 `foreach ss in 5 30 100` 对每个样本量、内层 `forvalues i=1/100` 对每个 μ 取值：把当前 μ 存入临时标量、调 `compute_power`、把返回的功效填入第 i 行。最后 `twoway` 用三种线型画三条功效曲线（样本量越大曲线越陡、越接近“阶跃”形状），并加变量标签、`graph export` 导出。图中可以读出：当真实 μ 偏离原假设 μ0=0 越多、样本量越大时，检验拒绝原假设的概率越接近1。


In [ ]:
%%stata
set obs 100
gen mu=-3+(_n-1)*6/99
tempname mu
gen power5=.
gen power30=.
gen power100=.
foreach ss in 5 30 100{
    forvalues i= 1/100{
        ** mu in [-3,3]
        local `mu'=-3+(`i'-1)*6/99
	compute_power, mu(``mu'') obs(`ss')
	replace power`ss'=r(power) in `i'
    }
}
label variable mu "Ture value of mu"
label variable  power5 "Power 5 Samples"
label variable  power30 "Power 30 Samples"
label variable  power100 "Power 100 Samples"
twoway (line power5 mu, lpattern(solid))/*
	*/ (line power30 mu, lpattern(dash))/*
	*/ (line power100 mu, lpattern(dot))
